# 零基础预备课：Python 与 PyTorch 的第一步

这节课专门为几乎没有编程基础的学习者准备。目标不是一次学完 Python，而是掌握后续语音识别课程马上会用到的最小知识。

完成后，你应当能：

1. 看懂变量、列表、函数调用和 `print`；
2. 创建 PyTorch Tensor，并解释 `shape` 与 `dtype`；
3. 看懂语音数据常见的 `[B, T, F]`；
4. 说出一次训练包含预测、误差、反向传播和更新。

> 学法：每个代码格先猜输出，再按 `Shift + Enter` 运行。猜错是最有价值的学习证据。

## 1. 变量：给数据起名字

`=` 在这里表示“把右边的值交给左边的名字”，不是数学里的等式证明。字符串要放在引号中。

In [ ]:
sample_rate = 16000
duration_seconds = 2
num_samples = sample_rate * duration_seconds
message = "你好，PyTorch"

print(message)
print("采样点数量：", num_samples)

assert num_samples == 32000

### 停下来预测

如果把 `duration_seconds` 改为 3，`num_samples` 会是多少？先写下答案，再修改并运行。

关键规则：采样点数 = 采样率 × 时长。单位也要相乘：`次/秒 × 秒 = 次`。

## 2. 从列表到 Tensor

Python 列表用方括号保存一串值。Tensor 也是数字容器，但它擅长批量计算、自动求梯度以及在 GPU 上运行。

In [ ]:
import torch

python_list = [0.0, 0.2, 0.5, 0.2, 0.0, -0.2, -0.5, -0.2]
waveform = torch.tensor(python_list, dtype=torch.float32)

print("Tensor：", waveform)
print("shape：", waveform.shape)
print("dtype：", waveform.dtype)
print("最大绝对振幅：", waveform.abs().max().item())

assert waveform.shape == torch.Size([8])
assert waveform.dtype == torch.float32

### shape 到底是什么？

`shape` 描述每个轴有多长。上面的波形只有一个轴，轴上有 8 个采样点，所以 shape 是 `[8]`。

常见语音 Tensor：

- `[T]`：一条波形，`T` 是采样点数；
- `[B, T]`：一批波形，`B` 是一次处理的语音条数；
- `[B, T, F]`：一批声学特征，`F` 是每个时间帧的特征数。

字母只是轴的名字。任何时候看到 Tensor，第一反应都应当是：每个轴代表什么？

In [ ]:
batch = torch.stack([waveform, waveform * 0.5])
print("batch shape：", batch.shape)
print(batch)

assert batch.shape == (2, 8)  # 2 条语音，每条 8 个采样点

### 小练习：不要先运行

预测以下三个表达式的结果：

1. `batch[0].shape`
2. `batch[:, :3].shape`
3. `batch.mean()` 表示什么？

其中 `:` 表示该轴全部都要，`:3` 表示取索引 0、1、2。

In [ ]:
print(batch[0].shape)
print(batch[:, :3].shape)
print(batch.mean())

assert batch[0].shape == (8,)
assert batch[:, :3].shape == (2, 3)

### 交互实验：拖动三个轴

拖动 `batch_size`、`time_steps` 和 `feature_dim`，观察 Tensor 的 shape。先让自己预测，再看输出。

In [ ]:
from ipywidgets import IntSlider, FloatSlider, interact

def inspect_feature_shape(batch_size=2, time_steps=8, feature_dim=4):
    features = torch.zeros(batch_size, time_steps, feature_dim)
    print(f"shape = {tuple(features.shape)}")
    print(f"一共有 {features.numel()} 个数字")
    print(f"第 1 条语音的 shape = {tuple(features[0].shape)}")

shape_widget = interact(
    inspect_feature_shape,
    batch_size=IntSlider(value=2, min=1, max=4, description="B"),
    time_steps=IntSlider(value=8, min=2, max=20, description="T"),
    feature_dim=IntSlider(value=4, min=1, max=10, description="F"),
)

## 3. 神经网络训练究竟在做什么？

先用最简单的直线模型理解训练：

$$\hat{y}=wx+b$$

`x` 是输入，`w` 和 `b` 是模型要学的参数，`ŷ` 是预测。训练的目标是让预测靠近正确答案 `y`。

下面故意从错误的 `w=0`、`b=0` 开始，让模型从数据中学出接近 `w=2`、`b=1`。

### 交互实验：参数怎样改变预测？

拖动 `x`、`w` 和 `b`。观察 `w` 如何控制变化速度，`b` 如何整体平移预测。

In [ ]:
def show_linear_prediction(x=1.0, w=0.0, b=0.0):
    prediction = w * x + b
    target = 2 * x + 1
    error = prediction - target
    print(f"预测值 = {prediction:.2f}")
    print(f"正确值 = {target:.2f}")
    print(f"预测误差 = {error:.2f}")

linear_widget = interact(
    show_linear_prediction,
    x=FloatSlider(value=1.0, min=-2.0, max=4.0, step=0.5),
    w=FloatSlider(value=0.0, min=-1.0, max=3.0, step=0.1),
    b=FloatSlider(value=0.0, min=-1.0, max=2.0, step=0.1),
)

In [ ]:
torch.manual_seed(0)

x = torch.tensor([[0.0], [1.0], [2.0], [3.0]])
y = 2 * x + 1

model = torch.nn.Linear(in_features=1, out_features=1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = torch.nn.MSELoss()

for step in range(101):
    prediction = model(x)               # 1. 前向：产生预测
    loss = loss_fn(prediction, y)       # 2. 比较：计算误差
    optimizer.zero_grad()               # 3. 清除上一轮梯度
    loss.backward()                     # 4. 反向：计算参数该怎么改
    optimizer.step()                    # 5. 更新参数

    if step in [0, 1, 10, 100]:
        print(f"step={step:3d}, loss={loss.item():.6f}")

learned_w = model.weight.item()
learned_b = model.bias.item()
print(f"学到的 w={learned_w:.3f}, b={learned_b:.3f}")

assert loss.item() < 1e-3
assert abs(learned_w - 2.0) < 0.05
assert abs(learned_b - 1.0) < 0.1

### 把直线训练映射到语音识别

| 直线例子 | 语音识别 |
|---|---|
| 输入 `x` | 波形或 Log-Mel 特征 |
| 模型 | Conv、RNN 或 Transformer |
| 预测 `ŷ` | 每个时间位置的字符/词元概率 |
| 正确答案 `y` | 语音对应的文字 |
| MSE loss | 常见为 CTC loss 或交叉熵 |
| 更新 `w, b` | 更新模型中数百万个参数 |

模型不同，但训练循环的骨架没有变。现在不需要理解 CTC；第 10～14 课会把它拆开。

## 4. 离场小测（请闭卷回答）

1. `sample_rate = 16000` 中，变量名和值分别是什么？
2. 一条含 32000 个采样点的波形，shape 通常是什么？
3. 一个特征 Tensor 的 shape 是 `[4, 100, 80]`，三个轴各可能表示什么？
4. `dtype=torch.float32` 说明了什么？
5. 用自己的话说出训练循环的五步。

通关标准：前 4 题至少答对 3 题，第 5 题能说出“预测—误差—反向—更新”即可。然后进入第 1 课《声音与采样》。

### 即时判题（先独立作答）

下面四题可以自动检查。第 5 题是表达题，请把答案发给老师批改；它不能仅靠关键词可靠判分。

In [ ]:
from IPython.display import display, clear_output
from ipywidgets import Button, Dropdown, HTML, Output, Text, VBox

q1 = Text(description="变量名", placeholder="sample_rate = 16000 中的变量名")
q2 = Dropdown(description="波形 shape", options=["请选择", "(32000,)", "(1, 32000)", "32000"])
q3 = Dropdown(description="B 是多少", options=["请选择", "4", "100", "80"])
q4 = Dropdown(description="float32", options=["请选择", "32 位浮点数", "32 个数字", "shape 长度为 32"])
check_button = Button(description="检查答案", button_style="primary")
feedback = Output()

def check_foundation_quiz(_):
    checks = [
        (q1.value.strip() == "sample_rate", "第 1 题：变量名是 sample_rate。"),
        (q2.value == "(32000,)", "第 2 题：一维波形 shape 是 (32000,)。"),
        (q3.value == "4", "第 3 题：[B,T,F]=[4,100,80]，所以 B=4。"),
        (q4.value == "32 位浮点数", "第 4 题：float32 表示每个元素用 32 位浮点格式保存。"),
    ]
    with feedback:
        clear_output()
        score = sum(ok for ok, _ in checks)
        print(f"得分：{score}/4")
        for index, (ok, explanation) in enumerate(checks, start=1):
            print(("✓" if ok else "✗"), explanation)
        if score >= 3:
            print("客观题达标。请继续口头解释训练循环的五步。")
        else:
            print("先回看对应小节，修改答案后可以再次检查。")

check_button.on_click(check_foundation_quiz)
display(VBox([HTML("<b>第 0 课客观题自检</b>"), q1, q2, q3, q4, check_button, feedback]))

## 5. 本课只记住这四句话

1. Tensor 是装数字的多维容器。
2. shape 的每个数字都必须能解释它代表什么。
3. 神经网络用输入产生预测，用 loss 衡量预测有多错。
4. `backward()` 计算修改方向，`optimizer.step()` 真正更新参数。

如果某段代码看不懂，不要笼统地说“PyTorch 不懂”。先指出：是不懂 Python 语法、Tensor shape，还是训练步骤。问题越具体，排错越快。